# Who Wants to Be a Millionaire — llama-cpp-python backend

This notebook runs the competition client on Google Colab using **llama-cpp-python**  
(no Ollama required — the model runs directly inside the Python process via GGUF).

**Runtime**: set to **GPU → T4** (or better) before running.  
Runtime → Change runtime type → Hardware accelerator → GPU

## 1 — Environment setup

Installs llama-cpp-python with CUDA support, then all other dependencies.  
This cell takes ~3-5 minutes on first run.

In [ ]:
import subprocess, sys

# Check whether a GPU is visible
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NOT FOUND — set Runtime to GPU!')

# Pre-built CUDA wheel (no compilation needed)
print('\nInstalling llama-cpp-python (pre-built CUDA wheel)...')
!pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
    --quiet

# Remaining Python dependencies
print('Installing remaining dependencies...')
!pip install faster-whisper huggingface-hub requests ddgs \
             sentence-transformers trafilatura numpy scipy psutil python-dotenv --quiet

print('\nAll packages installed.')

## 2 — Mount Google Drive (optional but recommended)

Caches the GGUF model file on Drive so you don't re-download it every session.  
Skip this cell if you prefer to download fresh each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_MODEL_DIR = '/content/drive/MyDrive/llm_models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print('Model cache directory:', DRIVE_MODEL_DIR)

## 3 — Download GGUF model

We use `huggingface_hub.hf_hub_download` which respects a local cache dir.  
Change `REPO_ID` and `FILENAME` to any GGUF you want — examples:

| Model | repo_id | filename |
|---|---|---|
| Qwen2.5-7B-Q4 | `Qwen/Qwen2.5-7B-Instruct-GGUF` | `qwen2.5-7b-instruct-q4_k_m.gguf` |
| Gemma-3-4B-Q4 | `google/gemma-3-4b-it-GGUF` | `gemma-3-4b-it-q4_k_m.gguf` |
| Llama-3.1-8B-Q4 | `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF` | `Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf` |

A Q4_K_M 7-8B model fits comfortably in T4's 15 GB VRAM.

In [ ]:
from huggingface_hub import hf_hub_download
import os

# ── CONFIGURE THIS ──────────────────────────────────────────────────────────
REPO_ID  = 'bartowski/Qwen2.5-7B-Instruct-GGUF'
FILENAME = 'Qwen2.5-7B-Instruct-Q4_K_M.gguf'
MODEL_NAME = 'qwen2.5'
# ────────────────────────────────────────────────────────────────────────────

# Use Drive cache if mounted, otherwise Colab's local /root/.cache
DRIVE_MODEL_DIR = '/content/drive/MyDrive/llm_models'
cache_dir = DRIVE_MODEL_DIR if os.path.isdir(DRIVE_MODEL_DIR) else None

print(f'Downloading {FILENAME} from {REPO_ID}...')
print(f'Cache dir: {cache_dir or "~/.cache/huggingface"}')

MODEL_PATH = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    cache_dir=cache_dir,
)
print(f'\nModel ready at: {MODEL_PATH}')
print(f'Size: {os.path.getsize(MODEL_PATH) / 1e9:.2f} GB')

## 4 — Clone / upload the client code

The easiest way is to clone your fork of the repo.  
Replace the URL with your own if needed, or upload the `colab-llama/` folder manually.

In [ ]:

import os, sys

# 1. Clona l'intero repository da GitHub nell'ambiente Colab
!git clone https://github.com/SufienSadgal/Natural-Language-Processing-project-.git

# 2. Imposta il percorso puntando alla cartella appena scaricata
CODE_DIR = '/content/Natural-Language-Processing-project-/final notebook polimillionaire'

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.chdir(CODE_DIR)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

## 5 — Configure environment

Set credentials and model settings.  
Use Colab Secrets (🔑 icon) to store `USERNAME` and `PASSWORD` instead of hardcoding.

In [ ]:
import os

# ── API ──────────────────────────────────────────────────────────────────────
API_URL  = 'http://131.175.15.22:51111/'

# Load from Colab Secrets if available, otherwise fill in manually
try:
    from google.colab import userdata
    USERNAME = userdata.get('USERNAME')
    PASSWORD = userdata.get('PASSWORD')
    print('Credentials loaded from Colab Secrets.')
except Exception:
    USERNAME = 'suf'   # ← change if not using Secrets
    PASSWORD = 'NLP2026!'   # ← change if not using Secrets

# ── Model ────────────────────────────────────────────────────────────────────
# MODEL_PATH / MODEL_NAME were set in cell 3; set them as env vars so
# ollama_client.py picks them up via os.getenv()
os.environ['MODEL_PATH']    = MODEL_PATH
os.environ['MODEL_NAME']    = MODEL_NAME
os.environ['N_CTX']         = '8192'   # context window
os.environ['N_GPU_LAYERS']  = '-1'     # -1 = offload all layers to GPU
os.environ['N_THREADS']     = '4'      # CPU threads for non-GPU ops

# ── RAG ──────────────────────────────────────────────────────────────────────
os.environ['ENABLE_RAG'] = 'true'   # set to 'false' to disable RAG

print(f'API:   {API_URL}')
print(f'User:  {USERNAME}')
print(f'Model: {MODEL_NAME}  ({MODEL_PATH})')
print(f'n_ctx={os.environ["N_CTX"]}  n_gpu_layers={os.environ["N_GPU_LAYERS"]}')

## 6 — Login

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as {user.username!r}  (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

## 7 — Quick model test

Verify the model loads and answers correctly before running a full game.  
First call is slow (cold load); subsequent calls reuse the cached instance.

In [ ]:
from ollama_client import ask_gemma

question = (
    'A researcher plans a study to examine long-term confidence in the U.S. economy among '
    'the adult population. She obtains a simple random sample of 30 adults as they leave a '
    'Wall Street office building one weekday afternoon. All but two of the adults agree to '
    'participate in the survey. Which of the following conclusions is correct?'
)
options = {
    0: 'Selection bias makes this a poorly designed survey.',
    1: 'The high response rate makes this a well-designed survey.',
    2: 'A voluntary response study like this gives too much emphasis to persons with strong opinions.',
    3: 'Proper use of chance as evidenced by the simple random sample makes this a well-designed survey.',
}

answer, reasoning, resources = ask_gemma(question, options)
print(f'Answer option: {answer}')
print(f'Throughput   : {resources["tps"]:.1f} tok/s')
print(f'Time         : {resources["elapsed"]:.2f}s')
print(f'\nReasoning:\n{reasoning[:500]}')

## 8 — Play a single text game

Competition IDs: **0** = Entertainment · **1** = History & Politics · **2** = Science & Nature · **3** = Mathematics

In [ ]:
from ollama_client import ask_gemma
from rag import fetch_rag_context, should_use_rag
from wrong_answers import save_wrong_answer


def play_game(game, use_rag=True, username='unknown', competition_id=None):
    """Play one game session autonomously (text mode)."""
    all_resources = []

    while game.in_progress:
        question = game.current_question
        if not question:
            print('No question available. Game may have ended.')
            break

        print(f'\n--- Level {game.current_level} ---')
        print(f'Q: {question.text}')

        question_options = {int(opt.id): opt.text for opt in question.options}
        for oid, otxt in question_options.items():
            print(f'  {oid}: {otxt}')

        time_left = game.time_remaining
        if time_left:
            print(f'Time remaining: {time_left:.1f}s')

        # RAG
        context = None
        if use_rag and should_use_rag(question.text):
            context = fetch_rag_context(question.text, question_options, k=5)

        answer_input, reasoning, resources = ask_gemma(question.text, question_options, context=context)
        answer_id = int(answer_input)
        all_resources.append(resources)

        result = game.answer(answer_id)

        if result.correct:
            print(f'✓ CORRECT!  Earned: ${result.earned_amount:,.2f}')
            if result.game_over:
                print('🎉 CONGRATULATIONS! Game complete!')
        elif result.timed_out:
            print('⏰ TIMED OUT!')
            save_wrong_answer(
                username=username, competition_id=competition_id,
                level=game.current_level, question=question.text,
                options=question_options, reasoning=reasoning,
                answer_given=answer_input, rag_context=context,
                resources=resources, timed_out=True,
            )
        else:
            print(f'✗ WRONG!  Earnings locked at ${result.earned_amount:,.2f}')
            save_wrong_answer(
                username=username, competition_id=competition_id,
                level=game.current_level, question=question.text,
                options=question_options, reasoning=reasoning,
                answer_given=answer_input, rag_context=context,
                resources=resources, timed_out=False,
            )

    # Summary
    print(f'\n=== Game Summary ===')
    print(f'Reached level : {game.current_level}')
    print(f'Total earnings: ${game.earned_amount:,.2f}')
    if all_resources:
        n = len(all_resources)
        print(f'Avg time      : {sum(r["elapsed"] for r in all_resources)/n:.2f}s')
        print(f'Avg throughput: {sum(r["tps"] for r in all_resources)/n:.1f} tok/s')


# ── Run ──────────────────────────────────────────────────────────────────────
COMP_ID = 0   # 0=Entertainment, 1=History, 2=Science, 3=Math
USE_RAG = True

game = client.game.start(competition_id=COMP_ID)
play_game(game, use_rag=USE_RAG, username=USERNAME, competition_id=COMP_ID)

## 9 — Multi-run text experiment

Plays `RUNS` games in a row for the chosen competition and prints a summary table.

In [ ]:
import time, io, contextlib, sys

COMP_ID = 0    # 0=Entertainment, 1=History, 2=Science, 3=Math
RUNS    = 5
PAUSE   = 10   # seconds between games
USE_RAG = True

results = []   # (level_reached, earned)

for run in range(1, RUNS + 1):
    sys.__stdout__.write(f'\n[{run}/{RUNS}] Starting game...\n')
    sys.__stdout__.flush()

    game = client.game.start(competition_id=COMP_ID)

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        play_game(game, use_rag=USE_RAG, username=USERNAME, competition_id=COMP_ID)

    level   = game.current_level
    earned  = game.earned_amount
    results.append((level, earned))
    sys.__stdout__.write(f'  → Level {level}  ${earned:,.2f}\n')
    sys.__stdout__.flush()

    if run < RUNS:
        time.sleep(PAUSE)

# Summary
print(f"\n{'='*42}")
print(f"  SUMMARY  {RUNS} games  comp={COMP_ID}")
print(f"{'='*42}")
for i, (lv, ea) in enumerate(results, 1):
    print(f"  Game {i:>2}: level {lv:>2}   ${ea:>12,.2f}")
print(f"  {'─'*36}")
levels   = [r[0] for r in results]
earnings = [r[1] for r in results]
print(f"  Avg level  : {sum(levels)/len(levels):.1f}")
print(f"  Avg earned : ${sum(earnings)/len(earnings):>12,.2f}")
print(f"  Best       : ${max(earnings):>12,.2f}")
print(f"  Worst      : ${min(earnings):>12,.2f}")

## 10 — Single speech game

The server streams audio for each question/option.  
`faster-whisper` transcribes them locally before passing to the model.  
Whisper model downloads automatically on first call (~39 MB for `tiny`, ~142 MB for `base`).

In [ ]:
from speech_client import play_game_speech

COMP_ID = 0   # 0=Entertainment, 1=History, 2=Science, 3=Math

game = client.game.start(competition_id=COMP_ID, mode='speech')
play_game_speech(game, username=USERNAME, competition_id=COMP_ID, use_rag=True)

## 11 — Multi-run speech experiment

Same as cell 9 but uses speech mode.  
Console output is suppressed per game to keep the log clean.

In [ ]:
import time, io, contextlib, logging, sys
import speech_client
from speech_client import play_game_speech

COMP_ID = 0    # 0=Entertainment, 1=History, 2=Science, 3=Math
RUNS    = 5
PAUSE   = 15   # seconds between games

results = []
_orig_ipython = speech_client._IPYTHON_AVAILABLE

for run in range(1, RUNS + 1):
    sys.__stdout__.write(f'\n[{run}/{RUNS}] Starting speech game...\n')
    sys.__stdout__.flush()

    game = client.game.start(competition_id=COMP_ID, mode='speech')

    speech_client._IPYTHON_AVAILABLE = False   # suppress audio widgets in Colab
    logging.disable(logging.CRITICAL)
    with contextlib.redirect_stdout(io.StringIO()):
        play_game_speech(game, username=USERNAME, competition_id=COMP_ID, use_rag=True)
    logging.disable(logging.NOTSET)
    speech_client._IPYTHON_AVAILABLE = _orig_ipython

    level  = game.current_level
    earned = game.earned_amount
    results.append((level, earned))
    sys.__stdout__.write(f'  → Level {level}  ${earned:,.2f}\n')
    sys.__stdout__.flush()

    if run < RUNS:
        sys.__stdout__.write(f'  Pausing {PAUSE}s...\n')
        sys.__stdout__.flush()
        time.sleep(PAUSE)

# Summary
print(f"\n{'='*42}")
print(f"  SPEECH SUMMARY  {RUNS} games  comp={COMP_ID}")
print(f"{'='*42}")
for i, (lv, ea) in enumerate(results, 1):
    print(f"  Game {i:>2}: level {lv:>2}   ${ea:>12,.2f}")
print(f"  {'─'*36}")
levels   = [r[0] for r in results]
earnings = [r[1] for r in results]
print(f"  Avg level  : {sum(levels)/len(levels):.1f}")
print(f"  Avg earned : ${sum(earnings)/len(earnings):>12,.2f}")
print(f"  Best       : ${max(earnings):>12,.2f}")
print(f"  Worst      : ${min(earnings):>12,.2f}")

## 12 — Leaderboard

In [ ]:
COMP_ID = 0

lb = client.leaderboard.get(COMP_ID)
print(f'Leaderboard: {lb.competition.name}\n')
print(f'{"Rank":>4}  {"Username":<20}  {"Level":>5}  {"Score":>12}')
print('─' * 48)
for rank, entry in enumerate(lb.entries[:20], 1):
    marker = ' ◄' if entry.username == USERNAME else ''
    print(f'{rank:>4}  {entry.username:<20}  {entry.reached_level:>5}  {entry.score:>12,.2f}{marker}')